# NazarAI: re-select operating point & evaluate an existing run (no retraining)

Inputs: the output of the `nazarai-train` notebook (trained model + validation predictions), the `nazarai-ml-src`
package dataset and the resized EyePACS dataset (the validation images). Steps:
1. choose the referral threshold for the deployed rule from **validation predictions only**
   (`nazarai_ml.select_operating_point`);
2. evaluate exactly as served (ONNX Runtime, CPU, 4-view TTA) on validation and the external IDRiD set.

In [ ]:
import glob, json, os, shutil, sys
WORK = "/kaggle/working"
TARGET_SENSITIVITY = 0.90
!pip install -q onnxruntime huggingface_hub
hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/preprocess.py", recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), "select_operating_point.py"))]
assert hits, "attach the updated nazarai-ml-src dataset"
shutil.copytree(hits[0], f"{WORK}/src/nazarai_ml", dirs_exist_ok=True)
sys.path.insert(0, f"{WORK}/src"); os.environ["PYTHONPATH"] = f"{WORK}/src"; os.chdir(WORK)
run_src = os.path.dirname(sorted(glob.glob("/kaggle/input/**/run/model.onnx", recursive=True))[0])
man_src = os.path.dirname(sorted(glob.glob("/kaggle/input/**/manifests/val.csv", recursive=True))[0])
shutil.copytree(run_src, f"{WORK}/run", dirs_exist_ok=True)
shutil.copytree(man_src, f"{WORK}/manifests", dirs_exist_ok=True)
import pandas as pd
val = pd.read_csv(f"{WORK}/manifests/val.csv")
missing = [p for p in val.path.head(50) if not os.path.exists(p)]
print("run:", run_src, "| manifests:", man_src, "| val images reachable:", 50 - len(missing), "/ 50")
assert not missing, f"validation image paths not mounted, e.g. {missing[0]}"

In [ ]:
# external test images are re-exported to the same paths the manifest uses
!python -m nazarai_ml.idrid_hf --out {WORK}/idrid

In [ ]:
!python -m nazarai_ml.select_operating_point --run {WORK}/run --target-sensitivity {TARGET_SENSITIVITY}

In [ ]:
%%time
!python -m nazarai_ml.evaluate --run {WORK}/run --manifests {WORK}/manifests --reports {WORK}/reports

In [ ]:
m = json.load(open(f"{WORK}/reports/metrics.json"))
for k, s in m["splits"].items():
    r = s["referable"]
    print(f"{s['name']:28s} n={s['n_images']:5d}  QWK={s['qwk']:.3f}  sens={r['sensitivity']}  spec={r['specificity']}  AUC={r['auc']}")
os.makedirs(f"{WORK}/release", exist_ok=True)
for f in ["model.onnx", "model.pt", "model_config.json", "history.json", "val_predictions.csv"]:
    shutil.copy(f"{WORK}/run/{f}", f"{WORK}/release/{f}")
shutil.copytree(f"{WORK}/reports", f"{WORK}/release/reports", dirs_exist_ok=True)
shutil.rmtree(f"{WORK}/idrid", ignore_errors=True)   # keep the output small